In [9]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

PyTorch version: 2.4.1+cu121
CUDA available: True
Using device: cuda


In [10]:
features_norm = np.load("data/processed/features_norm.npy")
targets = np.load("data/processed/targets.npy")
train_idx = np.load("data/processed/train_idx.npy")
val_idx = np.load("data/processed/val_idx.npy")
test_idx = np.load("data/processed/test_idx.npy")
edge_indices = np.load("data/processed/edge_indices.npy", allow_pickle=True)
edge_weights = np.load("data/processed/edge_weights.npy", allow_pickle=True)
tickers = pd.read_csv("data/processed/tickers.csv")["0"].tolist()

print("features_norm:", features_norm.shape)
print("targets:", targets.shape)
print("train_idx:", train_idx.shape)
print("val_idx:", val_idx.shape)
print("test_idx:", test_idx.shape)
print("edge_indices:", len(edge_indices), "graphs")
print("tickers:", len(tickers))

features_norm: (2739, 462, 6)
targets: (2739, 462)
train_idx: (1490,)
val_idx: (252,)
test_idx: (997,)
edge_indices: 2739 graphs
tickers: 462


In [11]:
class StockGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers, dropout):
        super(StockGNN, self).__init__()

        self.convs = nn.ModuleList()
        self.convs.append(SAGEConv(in_channels, hidden_channels))
        for _ in range(num_layers - 1):
            self.convs.append(SAGEConv(hidden_channels, hidden_channels))

        self.output = nn.Linear(hidden_channels, out_channels)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, x, edge_index):
        for conv in self.convs:
            x = conv(x, edge_index)
            x = self.relu(x)
            x = self.dropout(x)
        x = self.output(x)
        return x.squeeze(-1)


# config
IN_CHANNELS     = 6
HIDDEN_CHANNELS = 64
OUT_CHANNELS    = 1
NUM_LAYERS      = 2
DROPOUT         = 0.2

model = StockGNN(IN_CHANNELS, HIDDEN_CHANNELS, OUT_CHANNELS, NUM_LAYERS, DROPOUT).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

StockGNN(
  (convs): ModuleList(
    (0): SAGEConv(6, 64, aggr=mean)
    (1): SAGEConv(64, 64, aggr=mean)
  )
  (output): Linear(in_features=64, out_features=1, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (relu): ReLU()
)

Total parameters: 9,153


In [12]:
def make_graph(t):
    x = torch.tensor(features_norm[t], dtype=torch.float32).to(device)
    edge_index = torch.tensor(edge_indices[t], dtype=torch.long).to(device)
    y = torch.tensor(targets[t], dtype=torch.float32).to(device)
    return x, edge_index, y


def run_epoch(idx, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    with torch.set_grad_enabled(is_train):
        for t in idx:
            x, edge_index, y = make_graph(t)
            pred = model(x, edge_index)
            loss = nn.functional.mse_loss(pred, y)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()

    return total_loss / len(idx)


# hyperparameters
LR          = 0.001
EPOCHS      = 100
PATIENCE    = 10

optimizer = optim.Adam(model.parameters(), lr=LR)

train_losses = []
val_losses   = []
best_val     = float("inf")
patience_counter = 0
best_weights = None

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_idx, optimizer)
    val_loss   = run_epoch(val_idx)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss < best_val:
        best_val = val_loss
        patience_counter = 0
        best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | Patience: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

# restore best weights
model.load_state_dict(best_weights)
print(f"\nBest val loss: {best_val:.6f}")

KeyboardInterrupt: 